# 02 — Обучение и выбор рекомендательной модели

## Постановка задачи

Для каждого известного пользователя ранжируем до 10 товаров, которых он ещё не видел в доступной истории. Основной сигнал — неявные события Retailrocket: просмотр (вес 1), добавление в корзину (3), покупка (5). Значения суммируются по паре пользователь–товар в обучающей истории.

Сравниваем Global Popularity, Recent Popularity, ItemKNN и библиотечную ALS из `implicit`. Подбор параметров ItemKNN и ALS ограничен хронологической validation-выборкой; финальный test в этом ноутбуке не загружается.

## Почему эти методы

- **Global Popularity** — понятный ориентир, который показывает, насколько задача объясняется популярностью товаров.
- **Recent Popularity** — baseline по предыдущим 14 дням до начала validation; он проверяет временной дрейф спроса, не заглядывая в будущее.
- **ItemKNN** — коллаборативная фильтрация «товар–товар»: похожесть строится по общим пользователям, а профиль пользователя суммирует соседние товары.
- **implicit ALS** — библиотечная факторизация матрицы взаимодействий с confidence-weighted objective. Модель обучается на данных проекта, она не предобученная.

Основная метрика — NDCG@10: она учитывает позиции рекомендаций. Recall@10 и Precision@10 измеряют охват релевантных товаров и точность выдачи, а Coverage@10 — долю каталога, которую модель вообще рекомендует. Уже просмотренные товары исключаются.

## Что реализовано в проекте, а что даёт библиотека

Popularity и ItemKNN — проектные классы в `src/recommendation.py`. ALS используется только из `implicit.als.AlternatingLeastSquares`; адаптер приводит библиотечный интерфейс к общему API проекта. NumPy, Pandas и SciPy используются как вычислительные инструменты; ALS применяется только из пакета `implicit`.

| Модель | Реализация проекта | Используемая библиотека |
|---|---|---|
| Global Popularity | `PopularityRecommender`: суммирует веса событий по товару | Pandas группирует события |
| Recent Popularity | `RecentPopularityRecommender`: учитывает предыдущие 14 дней | Pandas фильтрует события по времени |
| ItemKNN | `ItemKNNRecommender`: строит item-item сходство и оценивает профиль пользователя | SciPy выполняет операции над разреженными матрицами |
| implicit ALS | `ImplicitLibraryALSRecommender` — адаптер к `implicit.als.AlternatingLeastSquares` | Пакет `implicit` реализует и оптимизирует ALS; веса событий обучаются на train |

Здесь сравниваются не две реализации ALS, а готовая библиотечная ALS с популярностными ориентирами и ItemKNN.

## Хронологический протокол и границы бенчмарка

Порог истории пользователя (не менее пяти событий), когорта и ограниченный каталог выбранного режима (dev: до 20 000 пользователей и 1 500 товаров) определяются только по train. Модели получают события train этой когорты и каталога. Это ограниченный offline-бенчмарк, а не оценка всей аудитории или полного каталога.

Validation-цели фильтруются по тому же каталогу; релевантны новые для пользователя товары с событиями в validation. ItemKNN и ALS выбираются по NDCG@10 только на этой хронологической выборке. Test не читается и не участвует ни в подборе, ни в выборе конфигурации.

## Единый протокол и режим бенчмарка

Основной протокол во всех ноутбуках — **binary relevance**: любой новый товар с будущим событием релевантен (1), независимо от типа и повторов события. Macro NDCG@10, Recall@10 и Precision@10 считаются на одинаковых пользователях с новыми целями в фиксированном каталоге; Precision всегда делится на 10. Веса 1/3/5 остаются только сигналом обучения. Graded NDCG прежнего исследования хранится в `outputs/archive/pre_binary_unification/` и не смешивается с текущими результатами.

`BENCHMARK_MODE=dev` (по умолчанию): до 20 000 пользователей / 1 500 товаров. `full`: до 100 000 / 10 000. Это расширенный ограниченный бенчмарк, а не весь каталог. Когорта и каталог выбираются только по train. Внутренний обучающий срез ранжировщика использует собственную прошлую историю и всех подходящих пользователей, как в исходном исследовании. Полный режим пишет артефакты в `outputs/full/`, сохраняя dev. До запуска full выполните `python -m scripts.run_experiments --mode full --estimate-only`: оценка использует только train. Сетки гиперпараметров не расширяются.


In [ ]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd
import gc
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})
from IPython.display import display, Markdown

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "src" / "retailrocket.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the project directory.")
sys.path.insert(0, str(ROOT))
from src.benchmark import (
    get_benchmark_config, artifact_root, benchmark_manifest, validate_selection,
)
CONFIG = get_benchmark_config()
ARTIFACTS = artifact_root(ROOT, CONFIG)
PROCESSED = ROOT / "data" / "processed"
FIGURES = ARTIFACTS / "figures"
METRICS = ARTIFACTS / "metrics"
FIGURES.mkdir(parents=True, exist_ok=True)
METRICS.mkdir(parents=True, exist_ok=True)

from src.recommendation import (
    PopularityRecommender, RecentPopularityRecommender,
    ItemKNNRecommender, ImplicitLibraryALSRecommender,
    build_interaction_data, select_benchmark_users, select_catalog_items,
)
from src.evaluation import evaluate_recommender
from src.model_selection import (
    evaluate_recommender_grid, select_best_validation_configuration,
)
from src.retailrocket import load_processed_events

train = load_processed_events(PROCESSED / "train.csv")
validation = load_processed_events(PROCESSED / "validation.csv")
MIN_HISTORY = CONFIG.min_history
MAX_USERS = CONFIG.max_users
MAX_ITEMS = CONFIG.max_items
SEED = CONFIG.seed
K = CONFIG.k
RELEVANCE_MODE = CONFIG.relevance_mode


cohort = select_benchmark_users(train, min_history=MIN_HISTORY,
                                max_users=MAX_USERS, random_state=SEED)
catalog = select_catalog_items(train, max_items=MAX_ITEMS)
train_benchmark = train[
    train.visitorid.isin(cohort) & train.itemid.isin(catalog)
].copy()
validation_benchmark = validation[
    validation.visitorid.isin(cohort) & validation.itemid.isin(catalog)
].copy()
interaction_data = build_interaction_data(
    train_benchmark, user_ids=cohort, item_ids=catalog
)

print("Когорта, выбранная по train: {:,} пользователей; каталог: {:,} товаров.".format(
    len(cohort), len(catalog)))
print("Обучающих событий в бенчмарке: {:,}; ненулевых пар пользователь–товар: {:,}.".format(
    len(train_benchmark), interaction_data.matrix.nnz))
print("Событий validation в когорте и каталоге: {:,}.".format(len(validation_benchmark)))
assert train.event_time.max() < validation.event_time.min()
assert train_benchmark.visitorid.isin(cohort).all()
assert train_benchmark.itemid.isin(catalog).all()
manifest = benchmark_manifest(train, cohort, catalog, CONFIG)
(METRICS / "benchmark_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Режим:", CONFIG.mode, "| релевантность:", RELEVANCE_MODE)


## Подбор параметров на validation

Для уменьшения стоимости поиска используется компактная сетка: ItemKNN — `neighbors ∈ {25, 50, 100}` и `shrinkage ∈ {0, 10}` (6 комбинаций); implicit ALS — `factors ∈ {32, 64}`, `regularization ∈ {0.05, 0.1}`, `alpha ∈ {10, 20}` (8 комбинаций). Число итераций ALS зафиксировано на 8, seed — 42, чтобы поиск оставался небольшим и воспроизводимым. Для каждой конфигурации модель обучается только на train и оценивается по одной и той же validation-выборке; победитель определяется по NDCG@10.
Сетки и выбор конфигураций задаются здесь; `src/` содержит общие функции. Все варианты оцениваются с binary relevance.


In [ ]:
itemknn_grid = [
    {"neighbors": neighbors, "shrinkage": shrinkage}
    for neighbors in (25, 50, 100)
    for shrinkage in (0.0, 10.0)
]
als_grid = [
    {
        "factors": factors,
        "regularization": regularization,
        "alpha": alpha,
        "iterations": 8,
        "random_state": SEED,
        "num_threads": 1,
    }
    for factors in (32, 64)
    for regularization in (0.05, 0.1)
    for alpha in (10.0, 20.0)
]



## Эксперимент 1 — ItemKNN

**Гипотеза.** Увеличение соседства с 25 до 50 добавит полезные связи, но 100 соседей может добавить шум. Shrinkage=10 должен ослабить ненадёжное сходство по редким совместным взаимодействиям. Сравниваем полный факторный план при одинаковой истории и пользователях.

In [ ]:
itemknn_tuning = evaluate_recommender_grid(
    train_benchmark, validation_benchmark, cohort, catalog, itemknn_grid,
    lambda data, params: ItemKNNRecommender(**params).fit(data),
    model_name="ItemKNN", k=K, relevance_mode=RELEVANCE_MODE,
)

itemknn_tuning.to_csv(METRICS / "itemknn_tuning_validation.csv", index=False)
display(itemknn_tuning.round(6))
knn_surface = itemknn_tuning.pivot(index="neighbors", columns="shrinkage", values="ndcg@10")
fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(knn_surface.values, aspect="auto", cmap="Blues")
ax.set_xticks(np.arange(len(knn_surface.columns)), labels=[str(value) for value in knn_surface.columns])
ax.set_yticks(np.arange(len(knn_surface.index)), labels=[str(value) for value in knn_surface.index])
ax.set(xlabel="Shrinkage", ylabel="Число соседей", title="ItemKNN: validation NDCG@10")
for row in range(knn_surface.shape[0]):
    for column in range(knn_surface.shape[1]):
        ax.text(column, row, "{:.4f}".format(knn_surface.iloc[row, column]),
                ha="center", va="center", color="black")
fig.colorbar(image, ax=ax, label="NDCG@10")
fig.tight_layout()
fig.savefig(FIGURES / "itemknn_tuning_validation.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close("all")
plt.close(fig)



In [ ]:
knn_effects = itemknn_tuning.pivot(index="neighbors", columns="shrinkage", values="ndcg@10")
shrink_delta = knn_effects[10.0] - knn_effects[0.0]
display(shrink_delta.rename("Δ NDCG: shrinkage 10−0").to_frame())
neighbor_delta = knn_effects[10.0].diff()
display(Markdown(
    "**Вывод.** При фиксированном числе соседей shrinkage=10 меняет NDCG@10 на "
    + "; ".join(f"{n}: {v:+.5f}" for n, v in shrink_delta.items())
    + f". При shrinkage=10 переход 25→50 даёт {neighbor_delta.loc[50]:+.5f}, "
    + f"50→100 — {neighbor_delta.loc[100]:+.5f}. "
    + ("Shrinkage улучшает все проверенные варианты, что согласуется с подавлением шумных редких связей. "
       if (shrink_delta > 0).all() else "Эффект shrinkage неодинаков; его нельзя обобщать без учёта числа соседей. ")
    + "Большее соседство не гарантирует улучшения: дополнительные связи могут добавлять шум. "
    + "Таблица показывает локальную чувствительность; близкие точки без повторных seed/срезов не объявляем устойчиво различающимися."
))


## Эксперимент 2 — ALS

**Гипотеза.** 64 фактора вместо 32 дадут более выразительные профили. Увеличение regularization может сдержать переобучение, а alpha усилит доверие к наблюдавшимся взаимодействиям; на разреженной истории слишком сильное доверие может ухудшить ранжирование. Проверяем эффекты при фиксированных остальных параметрах, а не только средние по сетке. Итерации=8 и seed=42 фиксированы: сходимость и устойчивость к инициализации здесь не исследуются.

In [ ]:
als_tuning = evaluate_recommender_grid(
    train_benchmark, validation_benchmark, cohort, catalog, als_grid,
    lambda data, params: ImplicitLibraryALSRecommender(**params).fit(data),
    model_name="implicit ALS", k=K, relevance_mode=RELEVANCE_MODE,
)

als_tuning.to_csv(METRICS / "implicit_als_tuning_validation.csv", index=False)
display(als_tuning.round(6))
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True, constrained_layout=True)
for axis, alpha in zip(axes, sorted(als_tuning["alpha"].unique())):
    surface = als_tuning[als_tuning["alpha"] == alpha].pivot(
        index="regularization", columns="factors", values="ndcg@10"
    )
    image = axis.imshow(surface.values, aspect="auto", cmap="Greens",
                        vmin=als_tuning["ndcg@10"].min(),
                        vmax=als_tuning["ndcg@10"].max())
    axis.set_xticks(np.arange(len(surface.columns)), labels=[str(value) for value in surface.columns])
    axis.set_yticks(np.arange(len(surface.index)), labels=[str(value) for value in surface.index])
    axis.set(xlabel="Число факторов", title="ALS: alpha={}".format(alpha))
    for row in range(surface.shape[0]):
        for column in range(surface.shape[1]):
            axis.text(column, row, "{:.4f}".format(surface.iloc[row, column]),
                      ha="center", va="center", color="black")
axes[0].set_ylabel("Регуляризация")
fig.colorbar(image, ax=axes, label="Validation NDCG@10", shrink=0.85, pad=0.04)
fig.savefig(FIGURES / "implicit_als_tuning_validation.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close("all")
plt.close(fig)



In [ ]:
als_effect_rows = []
for parameter, low, high in [("factors", 32, 64), ("regularization", 0.05, 0.1), ("alpha", 10., 20.)]:
    controls = [p for p in ["factors", "regularization", "alpha"] if p != parameter]
    paired = als_tuning.pivot(index=controls, columns=parameter, values="ndcg@10")
    delta = paired[high] - paired[low]
    als_effect_rows.append({"parameter": parameter, "contrast": f"{high}−{low}",
                            "min_delta": delta.min(), "max_delta": delta.max(),
                            "positive_pairs": int((delta > 0).sum()), "pairs": len(delta)})
als_effects = pd.DataFrame(als_effect_rows)
display(als_effects.round(6))
for row in als_effect_rows:
    display(Markdown(f"**{row['parameter']}.** При прочих фиксированных параметрах изменение {row['contrast']} "
                     f"даёт ΔNDCG от {row['min_delta']:+.5f} до {row['max_delta']:+.5f}; "
                     f"улучшены {row['positive_pairs']} из {row['pairs']} пар."))
display(Markdown(
    "**Вывод.** Число факторов управляет ёмкостью профилей; доверие alpha и регуляризация меняют компромисс между подгонкой и обобщением. "
    "Знак эффекта проверяем по контролируемым парам выше: неодинаковые знаки означают взаимодействие параметров, а не универсальный оптимум. "
    "Восемь итераций и один seed фиксированы, поэтому эти эффекты не являются проверкой сходимости или устойчивости к инициализации. "
    "Recall и NDCG могут двигаться по-разному: больше найденных целей не гарантирует лучшие позиции."
))


## Эксперимент 3 — персонализация против популярности

**Гипотеза.** ItemKNN и ALS используют индивидуальную историю и превзойдут оба popularity baseline по NDCG@10. Проверим также цену улучшения по времени и Coverage. Конфигурации фиксируем по validation; эта же выборка использована при поиске, поэтому сравнение оптимистично и требует независимого test в ноутбуке 03.

In [ ]:
itemknn_tuning.to_csv(METRICS / "itemknn_tuning_validation.csv", index=False)
als_tuning.to_csv(METRICS / "implicit_als_tuning_validation.csv", index=False)

best_itemknn = select_best_validation_configuration(itemknn_tuning)
best_als = select_best_validation_configuration(als_tuning)
itemknn_params = {
    "neighbors": int(best_itemknn["neighbors"]),
    "shrinkage": float(best_itemknn["shrinkage"]),
}
als_params = {
    "factors": int(best_als["factors"]),
    "regularization": float(best_als["regularization"]),
    "alpha": float(best_als["alpha"]),
    "iterations": int(best_als["iterations"]),
    "random_state": SEED,
    "num_threads": 1,
}
selected_parameters = {
    "selection_metric": "ndcg@10",
    "relevance_mode": RELEVANCE_MODE,
    "benchmark": manifest,
    "random_seed": SEED,
    "validation_start": str(validation.event_time.min()),
    "itemknn": {
        "parameters": itemknn_params,
        "validation_ndcg@10": float(best_itemknn["ndcg@10"]),
    },
    "implicit_als": {
        "parameters": als_params,
        "validation_ndcg@10": float(best_als["ndcg@10"]),
    },
}
with (METRICS / "selected_model_params.json").open("w") as file:
    json.dump(selected_parameters, file, indent=2)

display(itemknn_tuning[["neighbors", "shrinkage", "ndcg@10", "recall@10", "fit_seconds"]].round(4))
display(als_tuning[["factors", "regularization", "alpha", "ndcg@10", "recall@10", "fit_seconds"]].round(4))

models = {}
fit_seconds = {}
def fit_timed(name, factory):
    started = time.perf_counter()
    model = factory()
    fit_seconds[name] = time.perf_counter() - started
    models[name] = model
    return model

fit_timed("Global Popularity", lambda: PopularityRecommender().fit(
    train_benchmark, catalog_items=catalog))
fit_timed("Recent Popularity (14d)", lambda: RecentPopularityRecommender().fit(
    train_benchmark, as_of=validation.event_time.min(), lookback_days=14,
    catalog_items=catalog))
fit_timed("ItemKNN", lambda: ItemKNNRecommender(**itemknn_params).fit(interaction_data))
fit_timed("implicit ALS", lambda: ImplicitLibraryALSRecommender(**als_params).fit(interaction_data))
fit_seconds["ItemKNN"] = float(best_itemknn["fit_seconds"])
fit_seconds["implicit ALS"] = float(best_als["fit_seconds"])

validation_summaries = []
validation_per_user = {}
for name, model in models.items():
    summary, per_user, _ = evaluate_recommender(
        model, validation_benchmark, train_benchmark, cohort, catalog, k=K, relevance_mode=RELEVANCE_MODE
    )
    summary["model"] = name
    summary["fit_seconds"] = fit_seconds[name]
    if name == "ItemKNN":
        summary.update(itemknn_params)
    elif name == "implicit ALS":
        summary.update(als_params)
    validation_summaries.append(summary)
    validation_per_user[name] = per_user

validation_results = pd.DataFrame(validation_summaries).set_index("model")
validation_results = validation_results.sort_values("ndcg@10", ascending=False)
validation_results.to_csv(METRICS / "model_validation_metrics.csv")
display(validation_results.round(4))
for name, frame in validation_per_user.items():
    frame.assign(model=name).to_csv(METRICS / ("validation_users_" + name.replace(" ", "_") + ".csv"), index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
validation_results[["ndcg@10", "recall@10"]].plot(kind="bar", ax=axes[0],
    color=["#4472C4", "#70AD47"])
axes[0].set(title="Качество ранжирования на validation", ylabel="Средняя метрика", xlabel="")
axes[0].tick_params(axis="x", rotation=20)
validation_results["coverage@10"].sort_values().plot(kind="barh", ax=axes[1], color="#ED7D31")
axes[1].set(title="Покрытие каталога на validation", xlabel="Уникальные рекомендованные товары / каталог", ylabel="")
fig.tight_layout()
fig.savefig(FIGURES / "model_validation_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close("all")
plt.close(fig)



In [ ]:
als_score = validation_results.loc["implicit ALS", "ndcg@10"]
knn_score = validation_results.loc["ItemKNN", "ndcg@10"]
recent_score = validation_results.loc["Recent Popularity (14d)", "ndcg@10"]
display(Markdown(
    f"**Вывод.** ALS: {als_score:.5f}, ItemKNN: {knn_score:.5f}, Recent Popularity: {recent_score:.5f}. "
    f"ALS минус ItemKNN: {als_score-knn_score:+.5f} ({100*(als_score/knn_score-1):+.1f}%). "
    f"Coverage ALS/ItemKNN: {validation_results.loc['implicit ALS', 'coverage@10']:.1%} / "
    f"{validation_results.loc['ItemKNN', 'coverage@10']:.1%}. "
    "Относительный прирост сопоставляем с абсолютной разницей и стоимостью обучения в таблице. "
    "Это сравнение после подбора на той же validation: его оптимизм не устраняется bootstrap."
))


In [ ]:
from src.experiment_analysis import paired_metric_comparison
uncertainty = paired_metric_comparison(validation_per_user, reference="ItemKNN", seed=SEED)
display(uncertainty.round(6))
uncertainty.to_csv(METRICS / "model_validation_paired_intervals.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(uncertainty["delta"], uncertainty["model"],
            xerr=[uncertainty["delta"] - uncertainty["ci_low"], uncertainty["ci_high"] - uncertainty["delta"]], fmt="o")
ax.axvline(0, color="gray", linestyle="--")
ax.set(xlabel="Разница NDCG@10 относительно ItemKNN; 95% bootstrap CI", title="Парное сравнение на validation")
plt.show()
plt.close("all")


Интервал, включающий ноль, означает, что знак разницы не установлен этим анализом; интервал выше нуля поддерживает положительный эффект на этой когорте. Bootstrap пересэмплирует пользователей совместно для обеих моделей. Он не учитывает зависимость пользователей через товары, перебор конфигураций и временной дрейф, поэтому validation-интервалы после подбора — диагностические. Практическую ценность оцениваем вместе с абсолютной разницей, покрытием и стоимостью обучения.



In [ ]:
for row in uncertainty.itertuples():
    supported = "интервал включает ноль; знак не установлен" if row.ci_low <= 0 <= row.ci_high else "интервал исключает ноль на этой когорте"
    display(Markdown(f"**{row.model} минус ItemKNN:** Δ={row.delta:+.5f}, 95% CI [{row.ci_low:+.5f}; {row.ci_high:+.5f}]; {supported}."))
